# FCM001 · 商品截面动量

每天收盘计算 60 日标准化动量（跳过不复权主连的换月日收益）与居中排名 g，下一交易日 09:00 按固定两多两空篮子执行。
配置只写 `entry_fraction=20%` 与 `exit_buffer_fraction=40%`，g 门槛由代码换算。当前结果单位是标准化组合收益，不是假设资金规模后的整数手数 PnL。

In [ ]:
import pandas as pd
from fcm001.dev.factor import g_thresholds, run_study
from infra.config import factor_strategies
from infra.runner import DEFAULT_END, DEFAULT_START, DEFAULT_WARMUP

STRATEGY = factor_strategies('FCM001')[0]
ENTRY_G, RETAIN_G = g_thresholds(STRATEGY)
display(pd.Series(STRATEGY))
pd.Series({'entry_g': ENTRY_G, 'retain_g': RETAIN_G})

## 1. 完整十年运行
读取 10 个商品十年 1min 与 daily（约 2–3 分钟），写入 `fcm001/runs/v3_10y/FCM001_daily_v1/`。

In [ ]:
result = run_study(
    strategy=STRATEGY,
    warmup_start=DEFAULT_WARMUP,
    start=DEFAULT_START,
    end_exclusive=DEFAULT_END,
)
result.summary

## 2. 每日排名、g 与目标篮子

In [ ]:
rankings = result.rankings
last_signal_date = rankings.loc[rankings['signal_valid'], 'trading_date'].max()
rankings[rankings['trading_date'].eq(last_signal_date)].sort_values('rank')

## 3. 组合收益、持仓和换仓记录

In [ ]:
result.pnl.set_index(pd.to_datetime(result.pnl['date']))['cumulative_net_return'].plot(
    figsize=(13, 4), title='FCM001 · cumulative normalized net return'
)
display(result.positions[result.positions['weight'].ne(0)].tail(20))
display(result.trades.tail(20))

## 4. 持有期、换手与缓冲区滞留
持有段按 positions 中连续同方向非零权重计算；缓冲区滞留指目标篮子成员已不在前二/后二、但仍在前四/后四。

In [ ]:
positions = result.positions.sort_values(['product_id', 'date']).copy()
positions['side'] = positions['weight'].gt(0).astype(int) - positions['weight'].lt(0).astype(int)
positions['spell'] = positions.groupby('product_id')['side'].transform(lambda s: s.ne(s.shift()).cumsum())
spells = (
    positions[positions['side'].ne(0)]
    .groupby(['product_id', 'spell'])
    .agg(side=('side', 'first'), start=('date', 'first'), end=('date', 'last'), holding_days=('date', 'size'))
    .reset_index()
)
display(spells.groupby('side')['holding_days'].describe())
display(spells.groupby('product_id').agg(spells=('spell', 'count'), mean_days=('holding_days', 'mean'), max_days=('holding_days', 'max')))

trades = result.trades.copy()
trades['year'] = pd.to_datetime(trades['trading_date']).dt.year
display(trades[trades['action'].eq('open')].groupby('year').size().rename('opens_per_year'))

members = rankings[rankings['target_leg'].ne('flat') & rankings['signal_valid']].copy()
members['in_entry_group'] = (members['g'] * members['target_weight'].gt(0).map({True: 1, False: -1})).ge(ENTRY_G - 1e-9)
members.groupby('target_leg')['in_entry_group'].value_counts(normalize=True).unstack()

## 5. 执行失败与数据检查

In [ ]:
pd.Series({
    'trade_rows': len(result.trades),
    'failed_rebalances': int((~result.pnl['rebalance_executed']).sum()),
    'average_daily_turnover': result.trades.groupby('trading_date')['weight_change'].sum().mean(),
    'minimum_valid_universe': rankings.groupby('trading_date')['universe_size'].first().min(),
    'days_below_minimum_universe': int(rankings.groupby('trading_date')['skip_reason'].first().notna().sum()),
})